<a href="https://colab.research.google.com/github/keyvanmahmoudi/Data_Science_Course/blob/main/solutions/Session04_groupby_merge_reshape_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۴: گروه‌بندی، ادغام و تغییر شکل جدول‌ها

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session04_groupby_merge_reshape.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

کتابخانه‌ها و داده‌هایی که پاسخ‌ها به آن‌ها نیاز دارند:

</div>

In [1]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/keyvanmahmoudi/Data_Science_Course/main/data/"
orders = pd.read_csv(DATA_URL + "orders_clean.csv", parse_dates=["order_date"])
customers = pd.read_csv(DATA_URL + "customers.csv", parse_dates=["signup_date"])
products = pd.read_csv(DATA_URL + "products.csv")
flights = pd.read_csv(DATA_URL + "flights.csv")

df = (orders.merge(products, on="product_id", how="left", validate="many_to_one")
            .merge(customers[["customer_id", "birth_year", "signup_date"]],
                   on="customer_id", how="left", validate="many_to_one"))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

پرفروش‌ترین کالای هر دسته (بر اساس مبلغ) چیست؟ (راهنما: `groupby` دوسطحی و بعد `idxmax`)

</div>

In [2]:
sales = df.groupby(["category", "product_name"])["total_price"].sum()
best = sales.loc[sales.groupby(level="category").idxmax()]
(best / 1e6).round(1).rename("revenue (million Toman)")

category     product_name   
Books        Python Handbook     54.2
Clothing     Sneakers           224.2
Electronics  Smart Watch        764.0
Grocery      Pistachio 500g     247.7
Home         Rug (small)        263.3
Name: revenue (million Toman), dtype: float64

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

سن هر مشتری را در زمان سفارش حساب کنید و میانگین مبلغ سفارش را به تفکیک دههٔ سنی (۲۰، ۳۰، ...) گزارش کنید.

</div>

In [3]:
# Iranian year 1403 starts in March 2024: approximate age with the Gregorian order year
df["age"] = df["order_date"].dt.year - df["birth_year"]
df["age_decade"] = (df["age"] // 10 * 10).astype(int)
df.groupby("age_decade")["total_price"].agg(orders="count", avg_basket="mean").round(0)

,orders,avg_basket
age_decade,,
10,14,3119071.0
20,411,2616304.0
30,254,2892280.0
40,246,2568866.0
50,268,2608019.0
60,3,2823333.0


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

برای هر مشتری، فاصلهٔ زمانی بین ثبت‌نام و **اولین** سفارش را به روز حساب کنید.

</div>

In [4]:
first_order = orders.groupby("customer_id")["order_date"].min().rename("first_order")
gap = customers.set_index("customer_id").join(first_order, how="inner")
gap["days_to_first_order"] = (gap["first_order"] - gap["signup_date"]).dt.days
print(gap["days_to_first_order"].describe().round(1))
print("customers whose first order is BEFORE their signup date:", (gap["days_to_first_order"] < 0).sum())

count    198.0
mean     205.4
std      177.2
min     -150.0
25%       83.8
50%      192.0
75%      351.0
max      578.0
Name: days_to_first_order, dtype: float64
customers whose first order is BEFORE their signup date: 34


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

بیشتر مشتری‌ها چند ماه بعد از ثبت‌نام اولین خریدشان را انجام داده‌اند. اما فاصلهٔ **منفی** برای بعضی مشتری‌ها یعنی سفارش قبل از ثبت‌نام! در دادهٔ واقعی این یک مشکل کیفیت داده است (مثلاً تاریخ ثبت‌نام بعداً با تاریخ تکمیل پروفایل جایگزین شده) و باید قبل از تحلیل، منشأ آن بررسی شود. در این دادهٔ ساختگی، تاریخ ثبت‌نام و سفارش مستقل از هم تولید شده‌اند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

جدولی بسازید که سطرهایش روزهای هفته و ستون‌هایش ساعت‌های روز باشد و تعداد سفارش‌ها را نشان دهد. پرترافیک‌ترین زمان کدام است؟

</div>

In [5]:
days = ["Saturday", "Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
table = pd.crosstab(pd.Categorical(df["order_date"].dt.day_name(), categories=days, ordered=True),
                    df["order_date"].dt.hour)
busiest = table.stack().idxmax()
print("busiest slot:", busiest, "with", table.stack().max(), "orders")
table

busiest slot: ('Saturday', np.int32(11)) with 22 orders


order_date,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22
row_0,,,,,,,,,,,,,,,
Saturday,7,8,11,22,9,13,10,16,17,9,13,10,7,10,10
Sunday,11,7,12,8,7,14,10,9,15,7,9,12,13,10,8
Monday,15,18,13,8,12,9,17,16,12,11,14,14,22,9,7
Tuesday,21,6,21,10,14,10,18,16,10,11,17,9,15,8,17
Wednesday,14,12,9,14,6,6,9,9,14,11,9,17,12,12,12
Thursday,12,8,12,11,7,8,11,7,13,10,8,11,8,15,9
Friday,10,5,14,9,14,10,7,12,14,11,9,17,4,11,9


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۵

در دادهٔ `flights`، رشد سالانهٔ کل مسافران را با `pct_change` حساب کنید. کدام سال بیشترین رشد را داشته است؟

</div>

In [6]:
yearly = flights.groupby("year")["passengers"].sum()
growth = (yearly.pct_change() * 100).round(1)
print(growth)
print("highest growth:", growth.idxmax(), f"({growth.max()}%)")

year
1949     NaN
1950    10.3
1951    21.8
1952    15.8
1953    14.2
1954     6.2
1955    18.9
1956    15.6
1957    12.2
1958     3.4
1959    12.4
1960    11.2
Name: passengers, dtype: float64
highest growth: 1951 (21.8%)
